In [1]:
from datasets import load_dataset, concatenate_datasets

data = load_dataset('hlt-lab/voicebench', 'mmsu')


# Get all the splits (e.g., 'law', 'engineering')
split_names = list(data.keys())
print(f"Found splits: {split_names}")

# Create a list of all the individual datasets
all_splits = [data[split] for split in split_names]

# Concatenate all the datasets into one
combined_dataset = concatenate_datasets(all_splits)


combined_dataset


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found splits: ['law', 'engineering', 'other', 'biology', 'business', 'economics', 'health', 'philosophy', 'psychology', 'history', 'chemistry', 'physics']


Dataset({
    features: ['audio', 'prompt', 'reference', 'question_id', 'cot_content', 'category', 'src'],
    num_rows: 3074
})

In [ ]:
combined_dataset[0]

In [ ]:
from datasets import load_from_disk

# prompt => prompt_for_tts

# requied by evaluate.
# 'key', 'question', 'options', 'answer_index', 'prompt_for_tts', 'audio', 'category'

ds = load_from_disk(os.environ.get("MMLU_REF_DS_PATH", "/path/to/your/dataset/MMLU_FULL/full_5k_hf_format.v1"))
ds

In [8]:
import re

def transform_example(example):
    """
    Transforms a single example from the original format to the new format.
    """
    # Extract question and options from the prompt
    prompt_text = example['prompt']
    
    # The question is the first line.
    # The options are the lines starting with A., B., C., D.
    # The last two lines are a generic instruction.
    lines = prompt_text.strip().split('\n')
    
    question = lines[0]
    
    # Find options (e.g., "A. ...", "B. ...")
    options = [line for line in lines if re.match(r'^[A-Z]\.\s', line)]
    
    # Create the prompt for text-to-speech, which includes the question and options
    prompt_for_tts = prompt_text
    
    # Remove the "A. ", "B. " prefixes for the options list
    cleaned_options = [re.sub(r'^[A-Z]\.\s', '', opt) for opt in options]
    
    # Convert the reference character ('A', 'B', etc.) to a zero-based index
    answer_index = ord(example['reference']) - ord('A')

    return {
        'key': example['src'] + "_" + str(example['question_id']),
        'question': question,
        'options': cleaned_options,
        'answer_index': answer_index,
        'prompt_for_tts': prompt_for_tts,
        'category': example['category']
    }

transformed_dataset = combined_dataset.map(transform_example, writer_batch_size=200, remove_columns=["reference", "src", "cot_content", "prompt"])


Map: 100%|██████████| 3074/3074 [00:03<00:00, 810.56 examples/s]


In [9]:
transformed_dataset[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.]),
  'sampling_rate': 16000},
 'question_id': 886,
 'category': 'law',
 'key': 'ori_mmlu-jurisprudence_886',
 'question': "Which of the following purposes does the 'internal point of view' play in Hart's concept of law?",
 'options': ['It distinguishes social rules from mere group habits.',
  'It defines the judicial function.',
  'It illustrates the authority of the legislature.',
  'It stresses the relationship between law and justice.'],
 'answer_index': 0,
 'prompt_for_tts': "Which of the following purposes does the 'internal point of view' play in Hart's concept of law?\nA. It distinguishes social rules from mere group habits.\nB. It defines the judicial function.\nC. It illustrates the authority of the legislature.\nD. It stresses the relationship between law and justice.\n\nWhat is the answer to the above multiple choice question? Select one of the following: A, B, C, or D."}

In [10]:
transformed_dataset.save_to_disk(os.environ.get("MMSU_DS_PATH", "/path/to/your/dataset/VB_MMSU/full_3074_hf_format.v0"))

Saving the dataset (5/5 shards): 100%|██████████| 3074/3074 [00:16<00:00, 188.92 examples/s]
